# LLM-Adapted Time Series Forecaster — Kaggle / Colab / Local Pipeline

Frozen-LLM time series forecasting: a contrastively-aligned patch encoder feeds a
frozen GPT-2, steered by a CMA-ES-optimized soft prompt, with a small MLP head
reading out the forecast.

**This notebook runs unchanged in three places** — the next cell detects which:

| Environment | GPU setup | Code |
|---|---|---|
| **Kaggle** | Settings → Accelerator → **GPU T4 x2**, **Internet On** | cloned from GitHub |
| **Colab** | Runtime → Change runtime type → **GPU** (T4 or better) | cloned from GitHub |
| **Local** | your own CUDA GPU | uses the repo you launched Jupyter from |

Then run all cells top-to-bottom. The ETT dataset (Electricity Transformer
Temperature) downloads automatically in Phase 1 from
[zhouhaoyi/ETDataset](https://github.com/zhouhaoyi/ETDataset) — no uploads needed.

When 2+ GPUs are visible (e.g. Kaggle T4 x2), training uses `DataParallel` across them.

**Local quick start:**
```bash
git clone https://github.com/211abhi/LLM-Adapted-Time-Series-Forecaster.git
cd LLM-Adapted-Time-Series-Forecaster
pip install -r requirements.txt
jupyter lab pipeline.ipynb
```

## 0. Setup — Detect Environment, Locate Code, Install Dependencies

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL  = 'https://github.com/211abhi/LLM-Adapted-Time-Series-Forecaster.git'
REPO_NAME = 'LLM-Adapted-Time-Series-Forecaster'


def detect_env() -> str:
    """'kaggle' | 'colab' | 'local' — checked in that order."""
    if os.path.exists('/kaggle'):
        return 'kaggle'
    try:
        import google.colab  # noqa: F401
        return 'colab'
    except ImportError:
        return 'local'


def find_repo_root(start: Path) -> Path | None:
    """Walk up from `start` looking for this repo's marker files."""
    for d in [start, *start.parents]:
        if (d / 'src').is_dir() and (d / 'scripts').is_dir() and (d / 'setup.py').is_file():
            return d
    return None


ENV = detect_env()

if ENV == 'local':
    # Never clone or pull locally — you are already working in a checkout, and a
    # pull here would fight with your own uncommitted edits.
    CODE_DIR = find_repo_root(Path.cwd())
    if CODE_DIR is None:
        CODE_DIR = Path.cwd() / REPO_NAME
        if not CODE_DIR.is_dir():
            subprocess.run(['git', 'clone', REPO_URL, str(CODE_DIR)], check=True)
    CODE_DIR = str(CODE_DIR)
else:
    base = '/kaggle/working' if ENV == 'kaggle' else '/content'
    CODE_DIR = f'{base}/{REPO_NAME}'
    # The repo is private: authenticate with GITHUB_PAT from the platform's secret
    # store (Kaggle: Add-ons -> Secrets, Colab: key icon), then strip the token
    # back out of .git/config so it never lands in saved notebook output.
    try:
        if ENV == 'kaggle':
            from kaggle_secrets import UserSecretsClient
            _pat = UserSecretsClient().get_secret('GITHUB_PAT')
        else:
            from google.colab import userdata
            _pat = userdata.get('GITHUB_PAT')
    except Exception:
        _pat = os.environ.get('GITHUB_PAT')
    _url = REPO_URL.replace('https://', f'https://{_pat}@') if _pat else REPO_URL
    if not os.path.isdir(CODE_DIR):
        subprocess.run(['git', 'clone', _url, CODE_DIR], check=True)
        subprocess.run(['git', '-C', CODE_DIR, 'remote', 'set-url', 'origin', REPO_URL])
    else:
        subprocess.run(['git', '-C', CODE_DIR, 'pull', _url, 'main'])

os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR
os.environ['CODE_DIR']   = CODE_DIR
os.environ['PIPELINE_ENV'] = ENV

for d in ['data/raw', 'data/processed', 'data/splits', 'configs',
          'checkpoints/encoder', 'checkpoints/soft_prompt', 'checkpoints/pred_head',
          'logs']:
    os.makedirs(os.path.join(CODE_DIR, d), exist_ok=True)

print(f'Environment : {ENV}')
print(f'Code dir    : {CODE_DIR}')
print(f'Working dir : {os.getcwd()}')


def ask(prompt: str, default: str, env_var: str | None = None) -> str:
    """input() that degrades gracefully.

    Kaggle "Save & Run All" and any headless `nbconvert --execute` have no stdin,
    where a bare input() raises and kills the run. Resolution order:
    environment variable -> interactive prompt -> default.
    """
    if env_var and os.environ.get(env_var):
        value = os.environ[env_var]
        print(f'{prompt}{value}   [from ${env_var}]')
        return value
    try:
        return input(prompt) or default
    except Exception:
        print(f'{prompt}{default}   [non-interactive; using default]')
        return default

## 0b. Secrets — GitHub PAT + Hugging Face Token (optional)

Both are **optional** — they are only needed if you want to push trained checkpoints
back to GitHub from the notebook. Each is read from the right place per environment,
so nothing is ever typed into a cell or saved into the notebook file:

| | Kaggle | Colab | Local |
|---|---|---|---|
| `GITHUB_PAT` | Add-ons → Secrets | 🔑 sidebar → Secrets | env var, or `.env` file |
| `HF_TOKEN` | Add-ons → Secrets | 🔑 sidebar → Secrets | env var, or `.env` file |

`GITHUB_PAT` is a classic PAT with `repo` scope. If none is found anywhere, the
checkpoint-push cells fall back to a hidden `getpass` prompt — and you can simply
skip those cells entirely. `HF_TOKEN` just avoids Hugging Face's unauthenticated
rate limit; gpt2-medium downloads fine without it.

In [ ]:
import os
from pathlib import Path

ENV = os.environ.get('PIPELINE_ENV', 'local')


def _from_kaggle(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


def _from_colab(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


def _from_dotenv(name):
    """Read KEY=value out of a local .env — never committed (see .gitignore)."""
    env_file = Path(os.environ.get('CODE_DIR', '.')) / '.env'
    if not env_file.is_file():
        return None
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line.startswith(f'{name}='):
            return line.split('=', 1)[1].strip().strip('"\'')
    return None


def get_secret(name: str):
    """Environment variable first, then the platform's secret store, then .env."""
    for source in (lambda n: os.environ.get(n), _from_kaggle, _from_colab, _from_dotenv):
        value = source(name)
        if value:
            return value
    return None


def get_github_token() -> str | None:
    token = get_secret('GITHUB_PAT')
    if token:
        return token
    try:
        from getpass import getpass
        return getpass('GitHub PAT (hidden): ') or None
    except Exception:  # headless run (Kaggle Save & Run All, nbconvert) — no stdin
        return None


_where = {'kaggle': 'Add-ons -> Secrets', 'colab': 'the key icon in the sidebar',
          'local': 'a GITHUB_PAT/HF_TOKEN env var or a .env file'}[ENV]

hf_token = get_secret('HF_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('HF_TOKEN found — authenticated Hugging Face Hub requests enabled.')
else:
    print(f'No HF_TOKEN found (set one via {_where}) — '
          'proceeding unauthenticated (fine for gpt2-medium, just rate-limited).')

print('GITHUB_PAT found.' if get_secret('GITHUB_PAT') else
      f'No GITHUB_PAT found (set one via {_where}) — checkpoint-push cells will prompt, '
      'or just skip them.')


def push_checkpoint(rel_paths, commit_msg: str, token: str | None = None) -> None:
    """Commit + push checkpoint file(s) to GitHub, from any environment.

    Pushes to a one-off tokenized URL instead of `git remote set-url`, so a local
    checkout's remote is never rewritten to embed your PAT.
    """
    import subprocess
    from pathlib import Path

    CODE_DIR = os.environ.get('CODE_DIR', os.getcwd())
    ENV      = os.environ.get('PIPELINE_ENV', 'local')

    if isinstance(rel_paths, (str, Path)):
        rel_paths = [rel_paths]
    rels = []
    for p in rel_paths:
        p = Path(p)
        p = p if p.is_absolute() else Path(CODE_DIR) / p
        if not p.exists():
            print(f'  (skip push — {p} not found)')
            continue
        rels.append(str(p.relative_to(CODE_DIR)))
    if not rels:
        return

    token = token or get_github_token()
    if not token:
        print('  (skip push — no GITHUB_PAT; download checkpoints from the run output instead)')
        return
    push_url = f'https://{token}@github.com/211abhi/LLM-Adapted-Time-Series-Forecaster.git'

    def git(*args, **kw):
        return subprocess.run(['git', '-C', CODE_DIR, *args], **kw)

    git('config', 'user.email', f'training@{ENV}')
    git('config', 'user.name', ENV.capitalize())
    git('add', '-f', *rels, check=True)
    if git('diff', '--cached', '--quiet').returncode == 0:
        print('  (nothing to push — checkpoint unchanged)')
        return
    git('commit', '-m', commit_msg, check=True)
    git('pull', '--rebase', push_url, 'main', check=True)
    git('push', push_url, 'HEAD:main', check=True)
    print('  pushed: ' + ', '.join(rels))

In [ ]:
import os, subprocess, sys

ENV      = os.environ.get('PIPELINE_ENV', 'local')
CODE_DIR = os.environ.get('CODE_DIR', os.getcwd())

PKGS = ['transformers>=4.40.0', 'datasets', 'ta', 'cma',
        'scikit-learn', 'pyyaml', 'tqdm', 'matplotlib', 'seaborn']


def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)


if ENV == 'kaggle':
    # Kaggle images ship a CUDA torch already — reinstalling it breaks the image.
    pip(*PKGS)
elif ENV == 'colab':
    pip(*PKGS)
else:
    # Local: install torch yourself for YOUR CUDA version (pytorch.org/get-started),
    # then this fills in the rest from requirements.txt.
    try:
        import torch  # noqa: F401
        print('torch already installed — installing remaining requirements only.')
    except ImportError:
        print('torch NOT found. Install the build matching your CUDA/ROCm/CPU setup '
              'from https://pytorch.org/get-started/locally/ , then re-run this cell.')
    req = os.path.join(CODE_DIR, 'requirements.txt')
    pip('-r', req) if os.path.isfile(req) else pip(*PKGS)

print('Dependencies ready.')

In [ ]:
import torch

if torch.cuda.is_available():
    device  = 'cuda'
    n_gpu   = torch.cuda.device_count()
    print(f'Device : cuda ({n_gpu} GPU{"s" if n_gpu > 1 else ""})')
    for i in range(n_gpu):
        p = torch.cuda.get_device_properties(i)
        print(f'  [{i}] {p.name} — {p.total_memory / 1e9:.1f} GB VRAM')
    if n_gpu > 1:
        print('  Multi-GPU detected — training will use DataParallel.')
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    device = 'mps'
    print('Device : mps (Apple Silicon) — expect this to be much slower than CUDA.')
else:
    device = 'cpu'
    print('Device : cpu — training the encoder and CMA-ES search will be VERY slow. '
          'Enable a GPU accelerator if you can.')

In [ ]:
import yaml, os

CODE_DIR = os.environ.get('CODE_DIR', os.getcwd())

# Same schema for every ETT variant (date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT) — only
# the source file and sampling frequency change. Add more here to extend the sweep.
DATASET_VARIANTS = {
    'ETTh1': {'source_url': 'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv', 'freq': '1h'},
    'ETTh2': {'source_url': 'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh2.csv', 'freq': '1h'},
    'ETTm1': {'source_url': 'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv', 'freq': '15min'},
    'ETTm2': {'source_url': 'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm2.csv', 'freq': '15min'},
}

def build_config(variant: str, output_steps: int) -> dict:
    """Full pipeline config for one ETT variant.

    `encoder.checkpoint_path` is NEVER variant-scoped — the encoder is trained
    once (on whichever variant runs Phase 2 first) and reused/frozen for every
    other variant. Soft prompt, prediction head, processed data, splits, and
    logs each get their own `{variant}` subfolder so runs don't clobber each other.
    """
    spec = DATASET_VARIANTS[variant]
    return {
        'project': {
            'name': 'llm-adapted-time-series-forecaster',
            'seed': 42,
            'device': device,
            'log_dir': f'{CODE_DIR}/logs/{variant}/',
            'checkpoint_dir': f'{CODE_DIR}/checkpoints/',
        },
        'data': {
            'variant': variant,
            'raw_path': f'{CODE_DIR}/data/raw/{variant}.csv',
            'source_url': spec['source_url'],
            'freq': spec['freq'],
            'processed_dir': f'{CODE_DIR}/data/processed/{variant}/',
            'splits_dir': f'{CODE_DIR}/data/splits/{variant}/',
            'datetime_col': 'datetime',
            'feature_cols': ['open', 'high', 'low', 'close', 'volume'],
            'derived_features': ['returns', 'log_returns', 'hl_range', 'body_ratio'],
            'indicators': {'atr_period': 14, 'rsi_period': 14, 'ema_fast': 20, 'ema_slow': 50},
            'split_ratios': {'train': 0.70, 'val': 0.15, 'test': 0.15},
            'normalization': 'rolling_minmax',
            'norm_window': 500,
        },
        'regime_labeling': {
            'rsi_high': 55, 'rsi_low': 45, 'rsi_mid_low': 40, 'rsi_mid_high': 60,
            'ranging_atr_mult': 0.3, 'atr_spike_mult': 1.5, 'hl_range_spike_mult': 1.5,
            'breakout_lookback': 20, 'volume_spike_mult': 1.5,
        },
        'tokenizer': {
            'window_size': 96, 'patch_size': 16, 'n_patches': 6,
            'n_features': 9, 'patch_dim': 144,
            'train_stride': 1, 'val_stride': 8,
        },
        'encoder': {
            'embed_dim': 256, 'n_heads': 8, 'n_layers': 4, 'ffn_dim': 512,
            'dropout': 0.1, 'max_seq_len': 7, 'output_dim': 256,
            'checkpoint_path': f'{CODE_DIR}/checkpoints/encoder/best_encoder.pt',
        },
        'alignment': {
            'temperature': 0.07,
            'projection_dim': 1024,
            'n_regimes': 5,
            'regime_labels': ['TRENDING_UP', 'TRENDING_DOWN', 'RANGING', 'VOLATILE', 'BREAKOUT'],
            'n_negatives': 4,
        },
        'encoder_training': {
            'epochs': 50, 'batch_size': 64, 'lr': 1e-4, 'weight_decay': 1e-5,
            'warmup_steps': 200, 'lr_schedule': 'cosine', 'patience': 10, 'val_every': 1,
        },
        'llm': {
            'model_name': 'gpt2-medium',
            'hidden_dim': 1024,
            'freeze_all': True,
            'hidden_state_layer': -1, 'hidden_state_position': 0,
            'cache_dir': f'{CODE_DIR}/.cache/llm',
        },
        'soft_prompt': {
            'n_tokens': 32,
            'token_dim': 1024,
            'init_std': 0.01,
            'checkpoint_path': f'{CODE_DIR}/checkpoints/soft_prompt/{variant}/best_soft_prompt.npy',
        },
        'cmaes': {
            'subspace_dim': 500, 'popsize': 20, 'sigma0': 0.1, 'maxiter': 100,
            'early_stop_patience': 30, 'fitness_metric': 'val_mae',
            'eval_batch_size': 32, 'max_eval_batches': 32,
        },
        'prediction_head': {
            'input_dim': 1024,
            'hidden_dim': 256, 'output_steps': output_steps, 'dropout': 0.1,
            'checkpoint_path': f'{CODE_DIR}/checkpoints/pred_head/{variant}/best_pred_head.pt',
        },
        'pred_head_training': {
            'epochs': 100, 'batch_size': 32, 'lr': 1e-3, 'weight_decay': 1e-5,
            'patience': 10, 'loss': 'mse',
        },
        'evaluation': {
            'metrics': ['mae', 'rmse', 'directional_accuracy'],
            'per_regime': True, 'rolling_window_months': 1,
        },
    }

# ── User inputs ──────────────────────────────────────
output_steps = int(ask('Forecast steps? (1 step = 1 hour | 24=1d  48=2d  168=1wk) [default 24]: ',
                       '24', env_var='OUTPUT_STEPS'))
print(f'  → forecasting {output_steps} steps = {output_steps} hours ahead')

VARIANT = 'ETTh1'
cfg = build_config(VARIANT, output_steps)

cfg_path = f'{CODE_DIR}/configs/base_config.yaml'
os.makedirs(os.path.dirname(cfg_path), exist_ok=True)
with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f'Config written → {cfg_path}')
print(f'  device        : {device}')
print(f'  variant       : {VARIANT}')
print(f'  output_steps  : {output_steps} ({output_steps} hours)')
print(f'  llm hidden_dim: {cfg["llm"]["hidden_dim"]}')

## Phase 1 — Data Preparation

Cleans raw OHLCV, adds indicators, normalizes, and saves train/val/test splits.

In [ ]:
import sys
!{sys.executable} {CODE_DIR}/scripts/01_prepare_data.py --config {CODE_DIR}/configs/base_config.yaml

In [ ]:
# Quick sanity check
import pandas as pd, yaml, os
CODE_DIR = os.environ.get('CODE_DIR', os.getcwd())
with open(f'{CODE_DIR}/configs/base_config.yaml') as f:
    cfg = yaml.safe_load(f)
df = pd.read_csv(os.path.join(cfg['data']['processed_dir'], 'processed.csv'), parse_dates=['datetime'])
print(f'Processed rows: {len(df)}')
print(df.describe().round(3))

## Phase 1b — Regime Labeling

Assigns rule-based market regime labels (TRENDING_UP / DOWN / RANGING / VOLATILE / BREAKOUT).

In [ ]:
import sys
!{sys.executable} {CODE_DIR}/scripts/02_label_regimes.py --config {CODE_DIR}/configs/base_config.yaml

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, yaml, os
CODE_DIR = os.environ.get('CODE_DIR', os.getcwd())
with open(f'{CODE_DIR}/configs/base_config.yaml') as f:
    cfg = yaml.safe_load(f)
df = pd.read_csv(os.path.join(cfg['data']['processed_dir'], 'with_regimes.csv'), parse_dates=['datetime'])
counts = df['regime'].value_counts()
counts.plot(kind='bar', figsize=(8, 4), title='Regime Distribution (per hour)', color='steelblue')
plt.tight_layout()
plt.show()
print(counts)

## Phase 2 — Train TS Encoder (Contrastive)

Trains the 1D-Conv + Transformer encoder against frozen LLM text embeddings via InfoNCE loss.

⏱ ~20–40 min on T4 depending on dataset size.

In [ ]:
import sys
from pathlib import Path

_encoder_ckpt = Path(f'{CODE_DIR}/checkpoints/encoder/best_encoder.pt')
_force = ''
if _encoder_ckpt.exists():
    _ans = ask(f'Encoder checkpoint already exists at {_encoder_ckpt}. '
                 f'Continue training from it (warm start, not from scratch)? [y/N]: ', 'n', env_var='FORCE_RETRAIN').strip().lower()
    _force = '--force' if _ans == 'y' else ''

!{sys.executable} {CODE_DIR}/scripts/03_train_encoder.py --config {CODE_DIR}/configs/base_config.yaml {_force}

In [ ]:
from pathlib import Path
import os
CODE_DIR = os.environ.get('CODE_DIR', os.getcwd())
ckpt = Path(f'{CODE_DIR}/checkpoints/encoder/best_encoder.pt')
print('Encoder checkpoint saved:', ckpt.exists())
if ckpt.exists():
    print(f'Size: {ckpt.stat().st_size / 1e6:.1f} MB')

In [ ]:
# Save the encoder checkpoint to GitHub so you never need to retrain it.
# Optional — skip this cell if you don't have a GITHUB_PAT.
push_checkpoint('checkpoints/encoder/best_encoder.pt', 'Add trained encoder checkpoint')

## Phase 3 — CMA-ES Soft Prompt Optimization

Optimizes the (32, 768) soft prompt prefix in a 500-dim subspace using evolutionary search.

⏱ ~1–3 hours on T4 (300 generations × 20 candidates). Reduce `maxiter` in config to speed up.

**To speed up:** Edit `configs/base_config.yaml` → `cmaes.maxiter: 50` for a quick test run.

In [ ]:
import sys
import yaml
from pathlib import Path

with open(f'{CODE_DIR}/configs/base_config.yaml') as f:
    _cfg = yaml.safe_load(f)
_sp_ckpt = Path(_cfg['soft_prompt']['checkpoint_path'])
_force = ''
if _sp_ckpt.exists():
    _ans = ask(f'Soft prompt checkpoint already exists at {_sp_ckpt}. '
                 f'Continue the search from it (warm start, not from scratch)? [y/N]: ', 'n', env_var='FORCE_RETRAIN').strip().lower()
    _force = '--force' if _ans == 'y' else ''

!{sys.executable} {CODE_DIR}/scripts/04_optimize_soft_prompt.py --config {CODE_DIR}/configs/base_config.yaml {_force}

In [ ]:
import numpy as np, os
from pathlib import Path
CODE_DIR = os.environ.get('CODE_DIR', os.getcwd())
sp_path = Path(f'{CODE_DIR}/checkpoints/soft_prompt/best_soft_prompt.npy')
print('Soft prompt saved:', sp_path.exists())
if sp_path.exists():
    arr = np.load(str(sp_path))
    print(f'Shape: {arr.shape}  mean={arr.mean():.4f}  std={arr.std():.4f}')

In [ ]:
# Save the soft prompt to GitHub so you can resume from Phase 4 in a new session.
# Optional — skip this cell if you don't have a GITHUB_PAT.
import yaml, os

CODE_DIR = os.environ.get('CODE_DIR', os.getcwd())
with open(f'{CODE_DIR}/configs/base_config.yaml') as f:
    cfg = yaml.safe_load(f)

push_checkpoint(cfg['soft_prompt']['checkpoint_path'], 'Save soft prompt checkpoint')

## Phase 4 — Train Prediction Head

Trains only the 2-layer MLP on frozen encoder + frozen LLM + frozen soft prompt.

⏱ ~5–15 min on T4.

In [ ]:
import sys
import yaml
from pathlib import Path

with open(f'{CODE_DIR}/configs/base_config.yaml') as f:
    _cfg = yaml.safe_load(f)
_ph_ckpt = Path(_cfg['prediction_head']['checkpoint_path'])
_force = ''
if _ph_ckpt.exists():
    _ans = ask(f'Pred head checkpoint already exists at {_ph_ckpt}. '
                 f'Continue training from it (warm start, not from scratch)? [y/N]: ', 'n', env_var='FORCE_RETRAIN').strip().lower()
    _force = '--force' if _ans == 'y' else ''

!{sys.executable} {CODE_DIR}/scripts/05_train_pred_head.py --config {CODE_DIR}/configs/base_config.yaml {_force}

In [ ]:
from pathlib import Path
import os
CODE_DIR = os.environ.get('CODE_DIR', os.getcwd())
ph_path = Path(f'{CODE_DIR}/checkpoints/pred_head/best_pred_head.pt')
print('Pred head saved:', ph_path.exists())

In [ ]:
# Save the prediction head to GitHub.
# Optional — skip this cell if you don't have a GITHUB_PAT.
import yaml, os

CODE_DIR = os.environ.get('CODE_DIR', os.getcwd())
with open(f'{CODE_DIR}/configs/base_config.yaml') as f:
    cfg = yaml.safe_load(f)

push_checkpoint(cfg['prediction_head']['checkpoint_path'], 'Save pred head checkpoint')

## Phase 5 — Evaluation

Runs the full pipeline on the held-out test set and reports MAE, RMSE, and directional accuracy.

In [ ]:
import sys
!{sys.executable} {CODE_DIR}/scripts/06_evaluate.py --config {CODE_DIR}/configs/base_config.yaml

In [ ]:
import json, yaml, os
CODE_DIR = os.environ.get('CODE_DIR', os.getcwd())
with open(f'{CODE_DIR}/configs/base_config.yaml') as f:
    cfg = yaml.safe_load(f)
with open(os.path.join(cfg['project']['log_dir'], 'evaluation_results.json')) as f:
    results = json.load(f)

print('=== Overall ===')
for k, v in results['overall'].items():
    print(f'  {k}: {v:.4f}')

if 'per_regime' in results:
    print('\n=== Per Regime ===')
    for regime, m in sorted(results['per_regime'].items()):
        print(f"  {regime:15s}  MAE={m['mae']:.4f}  DirAcc={m['directional_accuracy']:.1f}%")

## Quick Inference — Single Prediction

Run this after all phases are complete to test the forecaster on a single window.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, yaml, os
from src.prediction.forecaster import Forecaster
from src.utils.data_loader import get_feature_columns

CODE_DIR = os.environ.get('CODE_DIR', os.getcwd())
with open(f'{CODE_DIR}/configs/base_config.yaml') as f:
    cfg = yaml.safe_load(f)

forecaster = Forecaster.from_config(cfg)

df_proc = pd.read_csv(os.path.join(cfg['data']['processed_dir'], 'with_regimes.csv'), parse_dates=['datetime'])
df_raw  = pd.read_csv(cfg['data']['raw_path'], parse_dates=['date'])
df_raw  = df_raw.sort_values('date').reset_index(drop=True)

feature_cols = get_feature_columns(cfg)
arr      = df_proc[feature_cols].values
W        = cfg['tokenizer']['window_size']
n_steps  = cfg['prediction_head']['output_steps']
norm_win = cfg['data']['norm_window']

window     = arr[-W - n_steps: -n_steps][np.newaxis]
preds_norm = forecaster.predict(window)[0]

raw_ot     = df_raw['OT'].values.astype(float)
pred_start = len(raw_ot) - n_steps
norm_slice = raw_ot[max(0, pred_start - norm_win): pred_start]
ot_min, ot_max = norm_slice.min(), norm_slice.max()

preds_temp  = preds_norm * (ot_max - ot_min) + ot_min
actual_temp = raw_ot[-n_steps:]

context_len = 72
hist_temp   = raw_ot[-n_steps - context_len: -n_steps]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(-context_len, 0), hist_temp,   color='gray',      linewidth=1.2, label=f'Historical ({context_len}h)')
ax.plot(range(n_steps),         actual_temp, 'o-', color='steelblue', linewidth=1.5, label='Actual')
ax.plot(range(n_steps),         preds_temp,  's--', color='tomato',   linewidth=1.5, label='Predicted')
ax.axvline(0, color='black', linestyle=':', linewidth=0.8)
ax.fill_between(range(n_steps), preds_temp, actual_temp, alpha=0.12, color='tomato')
ax.set_title(f"{cfg['data'].get('variant', 'ETT')} Oil Temperature — {n_steps}-Step Forecast ({n_steps}h steps)")
ax.set_xlabel('Step')
ax.set_ylabel('Oil Temperature (°C)')
ax.legend()
plt.tight_layout()
plt.show()

mae  = abs(preds_temp - actual_temp).mean()
rmse = np.sqrt(((preds_temp - actual_temp) ** 2).mean())
dacc = (np.sign(np.diff(preds_temp)) == np.sign(np.diff(actual_temp))).mean() * 100
print(f'MAE  : {mae:.2f} °C')
print(f'RMSE : {rmse:.2f} °C')
print(f'Dir  : {dacc:.1f}%')
print(f'Pred : {preds_temp.min():.2f}°C – {preds_temp.max():.2f}°C')
print(f'Act  : {actual_temp.min():.2f}°C – {actual_temp.max():.2f}°C')

## Save Checkpoints
Download trained weights before the session ends.

In [ ]:
import zipfile, os

CODE_DIR = os.environ.get('CODE_DIR', os.getcwd())
ENV      = os.environ.get('PIPELINE_ENV', 'local')

zip_dir  = {'kaggle': '/kaggle/working', 'colab': '/content'}.get(ENV, CODE_DIR)
zip_path = os.path.join(zip_dir, 'checkpoints.zip')

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, filenames in os.walk(os.path.join(CODE_DIR, 'checkpoints')):
        for fname in filenames:
            fpath = os.path.join(root, fname)
            zf.write(fpath, os.path.relpath(fpath, CODE_DIR))

print(f'Zipped checkpoints -> {zip_path}  ({os.path.getsize(zip_path) / 1e6:.1f} MB)')

if ENV == 'kaggle':
    print('Kaggle: find checkpoints.zip in the Output tab on the right panel.')
elif ENV == 'colab':
    from google.colab import files
    files.download(zip_path)
else:
    print('Local: checkpoints are already on disk at '
          f'{os.path.join(CODE_DIR, "checkpoints")} — the zip is just a portable copy.')

## Multi-Dataset Soft Prompt Sweep (optional)

The encoder trained above is frozen and shared across every dataset — it is **never**
retrained here. This section trains a separate **soft prompt + prediction head per ETT
variant** (same schema, different transformer / sampling frequency) and evaluates each
independently, then compares results.

Run the cells above at least once first (through Phase 2) so `checkpoints/encoder/best_encoder.pt`
exists. Each variant below reuses it; only Phases 1, 1b, 3, 4, 5 repeat per dataset.

⏱ Budget accordingly — this is roughly (Phase 1+1b+3+4+5) × number of datasets.

In [ ]:
import subprocess, sys, yaml, json, os
from pathlib import Path

CODE_DIR = os.environ.get('CODE_DIR', os.getcwd())

assert Path(f'{CODE_DIR}/checkpoints/encoder/best_encoder.pt').exists(), \
    'Train the encoder first (Phase 2, above) — the sweep reuses it and never retrains it.'

FORCE_RETRAIN = os.environ.get('FORCE_RETRAIN', '').lower() == 'y'  # or set True to continue training/optimizing every phase from its existing
                       # checkpoint (warm start, not from scratch) even if one is already found.
                       # Unattended — no per-variant prompt, since the sweep runs hands-off.

PUSH_CHECKPOINTS = bool(os.environ.get('GITHUB_PAT')) or get_secret('GITHUB_PAT') is not None
                       # set False to run the sweep without pushing anything to GitHub.


def run_phase(script: str, cfg_path: str, force: bool = False) -> None:
    cmd = [sys.executable, f'{CODE_DIR}/scripts/{script}', '--config', cfg_path]
    if force:
        cmd.append('--force')
    result = subprocess.run(cmd, cwd=CODE_DIR)
    if result.returncode != 0:
        raise RuntimeError(f'{script} failed for {cfg_path}')


def maybe_push(path: str, commit_msg: str, token) -> None:
    if not PUSH_CHECKPOINTS:
        return
    push_checkpoint(path, commit_msg, token)


def run_variant(variant: str, output_steps: int, token=None) -> dict:
    """Phases 1 -> 1b -> 3 -> 4 -> 5 for one dataset variant. Phase 2 (encoder)
    is intentionally never run here — it's shared/frozen, trained once above."""
    print(f"\n{'=' * 60}\n{variant}\n{'=' * 60}")
    cfg = build_config(variant, output_steps)
    cfg_path = f'{CODE_DIR}/configs/{variant}_config.yaml'
    with open(cfg_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)

    run_phase('01_prepare_data.py', cfg_path)
    run_phase('02_label_regimes.py', cfg_path)
    run_phase('04_optimize_soft_prompt.py', cfg_path, force=FORCE_RETRAIN)
    maybe_push(cfg['soft_prompt']['checkpoint_path'], f'Save {variant} soft prompt checkpoint', token)
    run_phase('05_train_pred_head.py', cfg_path, force=FORCE_RETRAIN)
    maybe_push(cfg['prediction_head']['checkpoint_path'], f'Save {variant} pred head checkpoint', token)
    run_phase('06_evaluate.py', cfg_path)

    with open(os.path.join(cfg['project']['log_dir'], 'evaluation_results.json')) as f:
        return json.load(f)

In [ ]:
SWEEP_VARIANTS = ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2']

token = get_github_token() if PUSH_CHECKPOINTS else None
sweep_results = {}
for variant in SWEEP_VARIANTS:
    sweep_results[variant] = run_variant(variant, output_steps, token)

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

# Reads evaluation_results.json straight off disk for each variant, so this
# works even if `sweep_results` isn't in memory (kernel restart, ran cells out
# of order, or you're revisiting this after the sweep loop already finished).
rows = []
for variant in DATASET_VARIANTS:
    cfg = build_config(variant, output_steps)
    results_path = Path(cfg['project']['log_dir']) / 'evaluation_results.json'
    if not results_path.exists():
        print(f'  (skip {variant} — no evaluation_results.json yet; run the sweep loop above first)')
        continue
    with open(results_path) as f:
        res = json.load(f)
    o = res['overall']
    rows.append({'dataset': variant, 'MAE': o['mae'], 'RMSE': o['rmse'], 'DirAcc %': o['directional_accuracy']})

comparison = pd.DataFrame(rows).set_index('dataset')
display(comparison.round(4))

comparison[['MAE', 'RMSE']].plot(kind='bar', figsize=(8, 4), title='Soft-Prompt-Adapted Forecast Error by Dataset')
plt.ylabel('Normalized error')
plt.tight_layout()
plt.show()